In [22]:
import numpy as np
import pandas as pd

df_train_raw = pd.read_csv("datasets/train.csv")
df_test_raw = pd.read_csv("datasets/test.csv")

df_train_raw.describe()

,batch_hour_seq,attr_tenure_m,attr_bpm,pub_hour_utc,vid_len_sec,val_watch_sec,val_shares_cnt,val_swipe_pct,post_payout_usd,target_fyp_viral
count,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000,8400.00000,8400.000000,8400.000000
mean,5.500000,30.902381,127.113095,12.026190,37.107143,-21.060908,15.952262,0.45774,61.968060,0.247619
std,2.872452,17.583505,28.181970,6.851844,16.782875,200.884237,5.481678,0.20476,123.431055,0.431655
min,1.000000,1.000000,80.000000,0.000000,15.000000,-999.000000,1.000000,0.00000,0.000000,0.000000
25%,3.000000,16.000000,100.000000,6.000000,15.000000,9.770000,12.000000,0.31775,0.000000,0.000000
50%,5.500000,32.000000,126.000000,12.000000,30.000000,17.510000,16.000000,0.45800,0.000000,0.000000
75%,8.000000,46.250000,153.000000,18.000000,45.000000,27.240000,19.000000,0.59600,0.000000,0.000000
max,10.000000,58.000000,174.000000,23.000000,60.000000,65.140000,41.000000,1.00000,449.420000,1.000000


In [23]:
df_dim_creadores = df_train_raw[df_train_raw["sys_traffic_src"] != "BOT_FARM"][["usr_key"]].drop_duplicates()

df_dim_audios = df_train_raw[["snd_key"]].drop_duplicates()

df_fact_telemetria = df_train_raw.drop(columns=["attr_tier", "attr_tenure_m", "attr_bpm"])

df_organico = df_fact_telemetria[df_fact_telemetria["sys_traffic_src"] == "ORGANIC"].replace(-999, np.nan)

df_cubo_olap = df_organico.groupby("vid_key").agg({
    "usr_key": "first",
    "snd_key": "first",
    "cat_code": "first",
    "pub_hour_utc": "first",
    "vid_len_sec": "first",
    "post_payout_usd": "first",
    "target_fyp_viral": "first",
    "val_shares_cnt": "sum",
    "val_watch_sec": "mean",
    "val_swipe_pct": "mean"
}).reset_index()




In [24]:
class EvaluadorKaggleFYP:
    @staticmethod
    def auditar_estrella(df_fact, df_dim_creadores, df_dim_audios):
        print("🔍 Evaluando Checkpoint 1: Esquema Estrella...")
        if not df_dim_creadores['usr_key'].is_unique: return "❌ ERROR: 'usr_key' tiene duplicados."
        if not df_dim_audios['snd_key'].is_unique: return "❌ ERROR: 'snd_key' tiene duplicados."
        
        columnas_prohibidas = ['attr_tier', 'attr_tenure_m', 'attr_bpm']
        for col in columnas_prohibidas:
            if col in df_fact.columns: return f"❌ ERROR: La columna descriptiva '{col}' sigue en fact_telemetria."
                
        print("✅ CHECKPOINT 1 SUPERADO: Normalización exitosa.\n")
        return True

    @staticmethod
    def auditar_cubo_olap(df_cubo):
        print("🔍 Evaluando Checkpoint 2: Cubo OLAP...")
        # En el set de entrenamiento hay ~700 videos orgánicos, en el de test hay ~300. 
        # Si la llave es única, el roll-up se hizo bien.
        if 'vid_key' not in df_cubo.columns or not df_cubo['vid_key'].is_unique:
            return "❌ ERROR: 'vid_key' no es único. No agrupaste a nivel de video."
            
        print("✅ CHECKPOINT 2 SUPERADO: Granularidad OLAP correcta.\n")
        return True

evaluador = EvaluadorKaggleFYP()


In [25]:
evaluador.auditar_estrella(df_fact_telemetria, df_dim_creadores, df_dim_audios)

evaluador.auditar_cubo_olap(df_cubo_olap)

🔍 Evaluando Checkpoint 1: Esquema Estrella...
✅ CHECKPOINT 1 SUPERADO: Normalización exitosa.

🔍 Evaluando Checkpoint 2: Cubo OLAP...
✅ CHECKPOINT 2 SUPERADO: Granularidad OLAP correcta.



True

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# --- TU CÓDIGO AQUÍ: Feature Engineering  ---
X_train = ...
y_train = df_cubo_olap['target_fyp_viral']
X_test_limpio = ... (Asegúrate de que tenga las mismas columnas que X_train y elimina la trampa de Leakage)

# 1. Entrenar el Modelo
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
modelo.fit(X_train, y_train)

# 2. Predecir sobre el set de prueba ciego
predicciones = modelo.predict(X_test_limpio)

# 3. Exportar a Kaggle
# submission = pd.DataFrame({
#     'vid_key': X_test_limpio['vid_key'], # Asegúrate de que tu matriz final aún tenga el ID o guárdalo antes
#     'target_fyp_viral': predicciones
# })
# submission.to_csv('submission.csv', index=False)
# print("¡Archivo listo! Súbelo al Leaderboard.")